In [43]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import nltk
from nltk.corpus import wordnet
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaConfig,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "data")

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

In [44]:
# Download WordNet resources for augmentation
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

In [45]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Torch: 2.2.2+cu118
CUDA available: True
GPU: NVIDIA L40S


In [46]:
pcl_df = pd.read_csv(
    DATA_PATH + '/dontpatronizeme_pcl.tsv', 
    delimiter='\t', 
    header=None)
pcl_df.columns = ['paragraph_id', 'article_id', 'keyword', 'country_code', 'paragraph', 'label']
pcl_df["y"] = pcl_df["label"].apply(lambda x: 0 if x < 2 else 1)
pcl_df['paragraph'] = pcl_df['paragraph'].fillna("").astype(str)
pcl_df['paragraph_id'] = pcl_df['paragraph_id']


In [47]:
train_ids = pd.read_csv( DATA_PATH + '/train_semeval_parids-labels.csv')
test_ids = pd.read_csv(DATA_PATH + '/dev_semeval_parids-labels.csv')

train_df = pd.merge(pcl_df, train_ids, how='right', left_on='paragraph_id', right_on='par_id')
test_df = pd.merge(pcl_df, test_ids, how='right', left_on='paragraph_id', right_on='par_id')


In [48]:
# Base training set before augmentation/balancing
train_base = train_df.reset_index(drop=True)
train_base = train_base.sample(frac=1, random_state=RNG_SEED)

print(train_base.shape)
print(train_base["y"].value_counts().sort_index())

(8375, 9)
y
0    7581
1     794
Name: count, dtype: int64


In [49]:
# Augment positive samples with WordNet synonym replacement
def _synonym_candidates(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            name = lemma.name().replace("_", " ").lower()
            if name != word.lower() and name.isalpha():
                syns.add(name)
    return list(syns)

def augment_text(text, replace_n=2):
    tokens = re.findall(r"[A-Za-z]+|[^A-Za-z]+", text)
    word_indices = [i for i, t in enumerate(tokens) if t.isalpha() and len(t) > 3]
    if not word_indices:
        return text
    random.shuffle(word_indices)
    replaced = 0
    for idx in word_indices:
        word = tokens[idx]
        candidates = _synonym_candidates(word)
        if candidates:
            tokens[idx] = random.choice(candidates)
            replaced += 1
        if replaced >= replace_n:
            break
    return "".join(tokens)

def build_train_df(augment_factor, replace_n, neg_pos_ratio=None):
    pos_df = train_base[train_base["y"] == 1].copy()
    augmented_rows = []
    for _ in range(augment_factor):
        aug = pos_df.copy()
        aug["paragraph"] = aug["paragraph"].apply(lambda x: augment_text(x, replace_n=replace_n))
        augmented_rows.append(aug)

    if augmented_rows:
        train_aug = pd.concat([train_base] + augmented_rows).sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)
    else:
        train_aug = train_base.copy()

    if neg_pos_ratio is None:
        train_final = train_aug
    else:
        pos_aug = train_aug[train_aug["y"] == 1].copy()
        neg_aug = train_aug[train_aug["y"] == 0].copy()
        neg_target = min(len(neg_aug), int(len(pos_aug) * neg_pos_ratio))
        neg_sampled = neg_aug.sample(n=neg_target, random_state=RNG_SEED)
        train_final = pd.concat([pos_aug, neg_sampled]).sample(frac=1, random_state=RNG_SEED).reset_index(drop=True)

    class_counts = train_final["y"].value_counts().sort_index()
    num_classes = len(class_counts)
    total = class_counts.sum()
    class_weights = torch.tensor(
        [total / (num_classes * class_counts[i]) for i in range(num_classes)],
        dtype=torch.float
    )

    return train_aug, train_final, class_weights

In [50]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["paragraph"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

def make_dataset(df):
    ds = Dataset.from_pandas(df[["paragraph", "y"]])
    ds = ds.map(tokenize, batched=True)
    ds = ds.rename_column("y", "labels")
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

dev_dataset = make_dataset(test_df)

/home/ubuntu/Desktop/NLP_IndividualCW/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 2094/2094 [00:00<00:00, 2452.52 examples/s]


In [51]:
config = RobertaConfig.from_pretrained(
    "roberta-base",
    num_labels=2,
    classifier_dropout=0.3,
    hidden_dropout_prob=0.2
 )

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    config=config
 )

/home/ubuntu/Desktop/NLP_IndividualCW/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [52]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainerCallback

AUGMENT_FACTOR_LIST = [0, 1, 2]
REPLACE_N_LIST = [1, 2, 3]
NEG_POS_RATIO_LIST = [None, 2, 3]  # None = no downsampling
LR_LIST = [1e-5, 2e-5, 3e-5]
EPOCH_LIST = [3, 4]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_positive": f1}

def best_threshold_for_probs(labels, probs):
    best_f1 = 0.0
    best_thresh = 0.5
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_f1, best_thresh

class ThresholdCallback(TrainerCallback):
    def __init__(self, trainer, dev_dataset):
        self.trainer = trainer
        self.dev_dataset = dev_dataset
        self.best_f1 = 0.0
        self.best_thresh = 0.5
        self.best_epoch = None

    def on_epoch_end(self, args, state, control, **kwargs):
        pred = self.trainer.predict(self.dev_dataset)
        probs = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()[:, 1]
        labels = pred.label_ids
        epoch_f1, epoch_thresh = best_threshold_for_probs(labels, probs)
        if epoch_f1 > self.best_f1:
            self.best_f1 = epoch_f1
            self.best_thresh = epoch_thresh
            self.best_epoch = int(state.epoch)
        print(
            f"Epoch {int(state.epoch)} best thresh {epoch_thresh:.2f} F1 {epoch_f1:.4f} | "
            f"best overall {self.best_f1:.4f} at {self.best_thresh:.2f}"
        )

results = []
run_id = 0

for aug_factor in AUGMENT_FACTOR_LIST:
    for replace_n in REPLACE_N_LIST:
        for neg_pos_ratio in NEG_POS_RATIO_LIST:
            train_aug, train_final, class_weights = build_train_df(
                augment_factor=aug_factor,
                replace_n=replace_n,
                neg_pos_ratio=neg_pos_ratio
            )
            train_dataset = make_dataset(train_final)

            for lr in LR_LIST:
                for epochs in EPOCH_LIST:
                    run_id += 1
                    print(
                        "\nRun", run_id,
                        "| aug", aug_factor, "replace", replace_n, "neg_ratio", neg_pos_ratio,
                        "| lr", lr, "epochs", epochs,
                        "| train", len(train_final), "dev", len(test_df)
                    )

                    training_args = TrainingArguments(
                        output_dir=f"./results_wce/run_{run_id}",
                        num_train_epochs=epochs,
                        per_device_train_batch_size=16,
                        per_device_eval_batch_size=16,
                        evaluation_strategy="epoch",
                        logging_strategy="epoch",
                        save_strategy="epoch",
                        load_best_model_at_end=False,
                        learning_rate=lr,
                        weight_decay=0.05,
                        warmup_ratio=0.1,
                        report_to="none",
                    )

                    model = RobertaForSequenceClassification.from_pretrained(
                        "roberta-base",
                        config=config
                    )

                    trainer = WeightedTrainer(
                        model=model,
                        args=training_args,
                        train_dataset=train_dataset,
                        eval_dataset=dev_dataset,
                        compute_metrics=compute_metrics,
                        class_weights=class_weights
                    )

                    threshold_cb = ThresholdCallback(trainer, dev_dataset)
                    trainer.add_callback(threshold_cb)

                    trainer.train()

                    dev_metrics = trainer.evaluate()
                    results.append({
                        "run_id": run_id,
                        "aug_factor": aug_factor,
                        "replace_n": replace_n,
                        "neg_pos_ratio": neg_pos_ratio,
                        "learning_rate": lr,
                        "epochs": epochs,
                        "eval_f1_positive": dev_metrics.get("eval_f1_positive"),
                        "best_thresh_f1": threshold_cb.best_f1,
                        "best_thresh": threshold_cb.best_thresh,
                        "best_epoch": threshold_cb.best_epoch,
                        "train_size": len(train_final),
                        "dev_size": len(test_df)
                    })

                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

results_sorted = sorted(results, key=lambda x: x["best_thresh_f1"], reverse=True)
print("\nTop 5 runs by best threshold F1:")
for row in results_sorted[:5]:
    print(row)

if results_sorted:
    best_run = results_sorted[0]
    best_run_dir = os.path.join("results_wce", f"run_{best_run['run_id']}")
    ckpt_dirs = sorted(
        [d for d in os.listdir(best_run_dir) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1])
    )
    best_ckpt = ckpt_dirs[-1] if ckpt_dirs else None
    print("\nBest run folder:", best_run_dir)
    print("Best run info:", best_run)
    if best_ckpt:
        print("Latest checkpoint:", os.path.join(best_run_dir, best_ckpt))
    else:
        print("No checkpoints found in best run folder")

Map: 100%|██████████| 8375/8375 [00:02<00:00, 2896.34 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Run 1 | aug 0 replace 1 neg_ratio None | lr 1e-05 epochs 3 | train 8375 dev 2094


Epoch,Training Loss,Validation Loss,F1 Positive
1,0.650000,0.383489,0.452830


Epoch 1 best thresh 0.85 F1 0.4916 | best overall 0.4916 at 0.85
